# Session 9: cleaning

Everything so far has used clean data, which was a kindness rather than
realism. Now you get the messy version of the same file.

**The one thing to take from today: every cleaning step is a decision, not a
rule.** Dropping the rows with no duration and filling them with the average
give different answers. Neither is wrong. What is wrong is not noticing that
you chose.

In [ ]:
import os
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "messy").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

pd.set_option("display.width", 120)

MESSY = "data/messy/plays_messy.csv"
messy = pd.read_csv(MESSY)
messy.head()

## First look

`shape`, `info`, `isna().sum()`. Always these three, in that order.

In [ ]:
messy.shape

**2,223 rows.** The clean file has 2,183. So there are 40 rows here that
should not exist, and we do not know yet what they are.

In [ ]:
messy.info()

In [ ]:
messy.isna().sum()

Four columns have gaps. And look carefully at the last column name:

```
minutes played
```

A space instead of an underscore, **and a trailing space after it**. Watch:

In [ ]:
print(repr(messy.columns[-1]))

In [ ]:
try:
    messy["minutes_played"]
except KeyError as error:
    print("KeyError:", error)

That is a five-minute confusion the first time it happens for real, because
the trailing space is invisible in every table view you will look at.
`repr()` on the column names is how you catch it.

## Find the problems before fixing any of them

`value_counts(dropna=False)` on every categorical column. This is the
session 6 `DISTINCT` habit, arriving where it matters.

In [ ]:
messy["genre"].value_counts(dropna=False).head(20)

**Four spellings of Electronic:** `Electronic`, `ELECTRONIC`, `electronic`,
and `"  Electronic "` with spaces around it.

To pandas those are four completely different genres. Every
`groupby("genre")` you have written this course would silently split them
into four rows and each one would be wrong.

In [ ]:
messy["device"].value_counts(dropna=False)

Same again, plus trailing spaces on some values.

`dropna=False` matters: without it the `NaN` row is hidden, and the missing
values are exactly what you are looking for.

## A trap worth knowing about before we start

The messy file has 49 countries written as the literal text `NA`. Where did
they go?

In [ ]:
print("country isna:      ", messy["country"].isna().sum())
print("country == 'NA':   ", (messy["country"] == "NA").sum())

In [ ]:
literal = pd.read_csv(MESSY, keep_default_na=False)
print("with keep_default_na=False, 'NA' strings:", (literal["country"] == "NA").sum())

`read_csv` treats a list of strings as missing **by default**, including
`NA`, `N/A`, `null`, `NaN`, and an empty cell. Helpful, most of the time.

And sometimes badly wrong. If `NA` in your data means **Namibia**, or North
America, pandas has just deleted a real value and called it missing. Same for
a product code like `NULL`.

The fix is to know your data. `keep_default_na=False` takes the file
literally, and `na_values=[...]` lets you say exactly what counts as missing.

Here `NA` really does mean "not recorded", so the default is what we want.

---

# Now clean it, one problem at a time

## 1. The column name

In [ ]:
df = messy.rename(columns={"minutes played ": "minutes_played"})
list(df.columns)

## 2. Text: strip first, then settle on one capitalisation

`.str` is how you reach the text of a whole column at once. The methods are
the ordinary string ones: `.str.strip()`, `.str.lower()`, `.str.title()`,
`.str.replace()`, `.str.contains()`.

In [ ]:
for column in ["artist_name", "track_name", "genre", "country", "device"]:
    df[column] = df[column].str.strip()

df["genre"] = df["genre"].str.title()
df["device"] = df["device"].str.lower()

print(sorted(df["genre"].dropna().unique()))
print(sorted(df["device"].dropna().unique()))

Eight genres and five devices, as it should be.

**Strip before you compare, always.** `" Pop "` and `"Pop"` are different
strings and the difference is invisible on screen.

Note that `.str.title()` is itself a choice: it would turn `"hip-hop"` into
`"Hip-Hop"`. Which capitalisation you standardise on is a decision like any
other.

## 3. Numbers that arrived as text

In [ ]:
print(df["minutes_played"].dtype)
[v for v in df["minutes_played"].dropna().unique() if not str(v).replace(".", "", 1).isdigit()][:6]

Some values have a `" min"` suffix, some use a space as a thousands
separator. Strip those out, then convert.

In [ ]:
df["minutes_played"] = (df["minutes_played"]
                        .str.replace(" min", "", regex=False)
                        .str.replace(" ", "", regex=False))

df["minutes_played"] = pd.to_numeric(df["minutes_played"], errors="coerce")

print(df["minutes_played"].dtype)
print("now missing:", df["minutes_played"].isna().sum())

**`errors="coerce"` is the important bit.** Anything it cannot read becomes
`NaN` instead of stopping your program. That turns a crash into a countable
problem, and counting it is how you decide what to do next.

Always check `.isna().sum()` after a conversion. If it jumped, your cleaning
just created missing values.

## 4. Dates: the one that matters

There are three formats in this column.

In [ ]:
raw = df["played_at"].str.strip()
iso = raw.str.match(r"^\d{4}-\d{2}-\d{2}$")
slash = raw.str.contains("/")
print("ISO yyyy-mm-dd: ", iso.sum())
print("dd/mm/yyyy:     ", slash.sum())
print("dd-mm-yyyy:     ", (~iso & ~slash).sum())

Here is the one-liner everybody reaches for. It runs without a word of
complaint.

In [ ]:
convenient = pd.to_datetime(raw, format="mixed", dayfirst=True)
print(convenient.head(3).tolist())
print("nothing failed to parse:", convenient.isna().sum())

Now the correct version: parse each format explicitly and combine.

In [ ]:
careful = (
    pd.to_datetime(raw, format="%Y-%m-%d", errors="coerce")
    .fillna(pd.to_datetime(raw, format="%d/%m/%Y", errors="coerce"))
    .fillna(pd.to_datetime(raw, format="%d-%m-%Y", errors="coerce"))
)
print("nothing failed to parse:", careful.isna().sum())

Both parsed every row. Both look fine. They disagree about 628 of them.

In [ ]:
disagree = convenient != careful
print("rows where the two disagree:", disagree.sum())

pd.DataFrame({
    "text": raw[disagree].head(5),
    "dayfirst=True": convenient[disagree].head(5),
    "parsed per format": careful[disagree].head(5),
})

**`dayfirst=True` is applied to every value, including the ISO ones.**

`2025-09-06`, which is the 6th of September, gets read as the 9th of June.
29% of the dates are now wrong, silently, and every monthly chart built on
them would be wrong too.

We can prove which one is right, because we happen to have the clean file.
In real life you would not, which is why you have to know your formats.

In [ ]:
clean_reference = pd.read_csv("data/clean/plays.csv", parse_dates=["played_at"])

# Rename the reference column first, or the merge produces played_at_x and
# played_at_y and the comparison below cannot find either of them.
reference = clean_reference[["play_id", "played_at"]].rename(
    columns={"played_at": "truth"}
)

check = (df.assign(convenient=convenient, careful=careful)
           .drop_duplicates(subset=["play_id"])
           .merge(reference, on="play_id"))

print("dayfirst=True correct for:     "
      f"{(check['convenient'] == check['truth']).mean() * 100:.1f}%")
print("parsed per format correct for: "
      f"{(check['careful'] == check['truth']).mean() * 100:.1f}%")

In [ ]:
df["played_at"] = careful

## 5. Duplicates

Look before you delete anything.

In [ ]:
print("duplicated rows:", df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values("play_id").head(4)

`keep=False` shows **all** copies rather than just the extras, so you can see
what is actually repeated.

These are genuinely accidental: identical right down to the `play_id`, which
is supposed to be unique. That last fact is what makes it safe to call them
duplicates. Without a unique id, two identical rows might be two real events.

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"{before} -> {len(df)}")

2,223 back to 2,183, which is exactly what the clean file has. A satisfying
moment, and a useful cross-check.

## 6. Missing values: the judgement call

Three options, three different answers, all three defensible.

In [ ]:
print(df.isna().sum()[lambda s: s > 0])

In [ ]:
dropped = df.dropna(subset=["minutes_played"])
filled = df.copy()
filled["minutes_played"] = filled["minutes_played"].fillna(
    filled["minutes_played"].mean()
)

print(f"drop the 94 rows:   {len(dropped)} rows, "
      f"{dropped['minutes_played'].sum():,.0f} minutes")
print(f"fill with the mean: {len(filled)} rows, "
      f"{filled['minutes_played'].sum():,.0f} minutes")

**Those two numbers differ by about 390 minutes, and both are honest.**

A rule of thumb worth having:

* **drop** when the missing value makes the row useless for your question
* **label** when the row is still useful without it
* **fill** only when you can defend the invented value out loud

For this dataset: a play with no duration cannot answer a question about
minutes, so those rows go. A play with no recorded device is still a real
play, so it stays, labelled.

In [ ]:
df = df.dropna(subset=["minutes_played"])
df["device"] = df["device"].fillna("unknown")
df["genre"] = df["genre"].fillna("Unknown")
df["country"] = df["country"].fillna("Unknown")

print(df.isna().sum().sum(), "missing values left")

## 7. Check what the cleaning threw away

Always. Every time. It is one line.

In [ ]:
lost = len(messy) - len(df)
print(f"before: {messy.shape}")
print(f"after:  {df.shape}")
print(f"{lost} rows removed ({lost / len(messy) * 100:.1f}% of the file)")

6% is fine. 60% would not be.

It is genuinely common to write a step that silently discards most of the
data. Here is the classic: `dropna()` with no `subset` drops a row if **any**
column is empty.

In [ ]:
reckless = messy.rename(columns={"minutes played ": "minutes_played"}).dropna()
print(f"bare dropna() would have kept {len(reckless)} of {len(messy)} rows")

That would have cost 238 rows instead of 94, for no reason at all.

So print the shape before and after. It is the difference between cleaning
your data and quietly deleting it.

## The cleaning, as one function

Now that we know the steps, here they are in the shape session 10 will want.
Notice that the comments say **why**, not what.

In [ ]:
def clean_plays(messy):
    """Return a cleaned copy of the messy listening log.

    Decisions made here, on purpose:
      * rows with no duration are dropped, because they cannot answer a
        question about minutes
      * rows with no device, genre or country are kept and labelled, because
        they are still real plays
      * dates are parsed per format, because a single dayfirst pass misreads
        every ISO date in the file
    """
    df = messy.rename(columns={"minutes played ": "minutes_played"}).copy()

    for column in ["artist_name", "track_name", "genre", "country", "device"]:
        df[column] = df[column].str.strip()
    df["genre"] = df["genre"].str.title()
    df["device"] = df["device"].str.lower()

    df["minutes_played"] = pd.to_numeric(
        df["minutes_played"].str.replace(" min", "", regex=False)
                            .str.replace(" ", "", regex=False),
        errors="coerce",
    )

    raw = df["played_at"].str.strip()
    df["played_at"] = (
        pd.to_datetime(raw, format="%Y-%m-%d", errors="coerce")
        .fillna(pd.to_datetime(raw, format="%d/%m/%Y", errors="coerce"))
        .fillna(pd.to_datetime(raw, format="%d-%m-%Y", errors="coerce"))
    )

    df = df.drop_duplicates().dropna(subset=["minutes_played"])

    df["device"] = df["device"].fillna("unknown")
    df["genre"] = df["genre"].fillna("Unknown")
    df["country"] = df["country"].fillna("Unknown")

    return df.reset_index(drop=True)


result = clean_plays(pd.read_csv(MESSY))
print(result.shape)
result.head(3)

## Next

`02-charts.ipynb`, where the clean data finally says something.